In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_merged = pd.read_csv('master_merged_data.csv')
df_merged.columns = df_merged.columns.str.upper()
df_merged = df_merged[df_merged['CASERNE'] != 79]
df_merged['RISK_LEVEL'] = df_merged['RISK_LEVEL'].astype('category')
df_incidents = df_merged[['YEAR', 'MONTH', 'CASERNE', 'INCIDENT_COUNT', 'RISK_LEVEL']]

print(df_incidents.shape)
df_incidents.head()

(14543, 5)


,YEAR,MONTH,CASERNE,INCIDENT_COUNT,RISK_LEVEL
0,2005,2,3,1,Low
1,2005,3,3,3,Low
2,2005,4,3,3,Low
3,2005,5,3,1,Low
4,2005,6,3,2,Low


In [ ]:
dict_casernes = {}
years = range(df_merged['YEAR'].min(), df_merged['YEAR'].max() + 1)
months = range(1, 13)

all_combos = pd.MultiIndex.from_product([years, months], names=['YEAR', 'MONTH'])
df_combos = pd.DataFrame(index=all_combos)

df_weather = pd.read_csv('weather-clean.csv')
df_weather.columns = df_weather.columns.str.upper()
df_weather = df_weather[['YEAR', 'MONTH', 'TEMPERATURE_MAX']]

for cas_no in df_incidents['CASERNE'].unique():
  # Put each set of caserne data into its own data frame
  dict_casernes[cas_no] = df_incidents[df_incidents['CASERNE'] == cas_no]
  # Make sure there's a record for each month
  dict_casernes[cas_no] = df_combos.merge(dict_casernes[cas_no], on=['YEAR', 'MONTH'], how='left')
  dict_casernes[cas_no]['CASERNE'] = dict_casernes[cas_no]['CASERNE'].fillna(cas_no)
  dict_casernes[cas_no]['INCIDENT_COUNT'] = dict_casernes[cas_no]['INCIDENT_COUNT'].fillna(0)
  dict_casernes[cas_no]['RISK_LEVEL'] = dict_casernes[cas_no]['RISK_LEVEL'].fillna('Low')
  # Convert MONTH to cyclical features
  dict_casernes[cas_no]['MONTH_SIN'] = np.sin(2 * np.pi * dict_casernes[cas_no]['MONTH'] / 12)
  dict_casernes[cas_no]['MONTH_COS'] = np.cos(2 * np.pi * dict_casernes[cas_no]['MONTH'] / 12)
  # Create a feature for the risk score of the previous month
  dict_casernes[cas_no]['LAG1_RISK_LEVEL'] = dict_casernes[cas_no]['RISK_LEVEL'].shift(1)
  dict_casernes[cas_no]['LAG1_INCIDENT_COUNT'] = dict_casernes[cas_no]['INCIDENT_COUNT'].shift(1)
  # Drop the first row cause it doesn't have a risk score for the previous month
  dict_casernes[cas_no] = dict_casernes[cas_no].dropna()
  # Merge weather data
  dict_casernes[cas_no] = dict_casernes[cas_no].merge(df_weather, on=['YEAR', 'MONTH'], how='inner')

In [ ]:
for cas_no in dict_casernes.keys():
  # Find the first year with a monthly incident count greater than zero
  year = dict_casernes[cas_no][dict_casernes[cas_no]['INCIDENT_COUNT'] > 0].iloc[0]['YEAR']
  # Drop all rows where incident count was zero the entire year
  if (year > 2005):
    dict_casernes[cas_no] = dict_casernes[cas_no][dict_casernes[cas_no]["YEAR"] >= year]

In [ ]:
static_features = [
    'CASERNE',
    'CRIME_COUNT',
    'TOTAL_BUILDING_AREA',
    'TOTAL_BUILDINGS',
    'RESIDENTIAL',
    'PUBLIC_SERVICES',
    'AVG_FLOORS'
]
df_static_features = df_merged[static_features]
df_static_features = df_static_features.drop_duplicates()

print(df_static_features.shape)
df_static_features.head()

(66, 7)


,CASERNE,CRIME_COUNT,TOTAL_BUILDING_AREA,TOTAL_BUILDINGS,RESIDENTIAL,PUBLIC_SERVICES,AVG_FLOORS
0,3,1645.0,1233617.0,10148.0,8572.0,4.0,1.100186
200,4,4114.0,2171551.0,6491.0,5931.0,39.0,1.639901
438,5,7626.0,2000604.0,7489.0,5557.0,14.0,1.366510
677,8,984.0,444293.0,1306.0,983.0,3.0,1.531703
887,9,7681.0,2922162.0,9985.0,9222.0,65.0,1.823455


In [ ]:
# Append all the caserne data frames into a single data frame.
df_merged = pd.concat(dict_casernes.values(), ignore_index=True)
# Merge static features data
df_merged = df_merged.merge(df_static_features, on=['CASERNE'], how='inner')
df_merged['CASERNE'] = df_merged['CASERNE'].astype('int')
print(df_merged.shape)
df_merged.head()

(15549, 16)


,YEAR,MONTH,CASERNE,INCIDENT_COUNT,RISK_LEVEL,MONTH_SIN,MONTH_COS,LAG1_RISK_LEVEL,LAG1_INCIDENT_COUNT,TEMPERATURE_MAX,CRIME_COUNT,TOTAL_BUILDING_AREA,TOTAL_BUILDINGS,RESIDENTIAL,PUBLIC_SERVICES,AVG_FLOORS
0,2005,2,3,1.0,Low,8.660254e-01,5.000000e-01,Low,0.0,3.6,1645.0,1233617.0,10148.0,8572.0,4.0,1.100186
1,2005,3,3,3.0,Low,1.000000e+00,6.123234e-17,Low,1.0,13.4,1645.0,1233617.0,10148.0,8572.0,4.0,1.100186
2,2005,4,3,3.0,Low,8.660254e-01,-5.000000e-01,Low,3.0,20.1,1645.0,1233617.0,10148.0,8572.0,4.0,1.100186
3,2005,5,3,1.0,Low,5.000000e-01,-8.660254e-01,Low,3.0,23.1,1645.0,1233617.0,10148.0,8572.0,4.0,1.100186
4,2005,6,3,2.0,Low,1.224647e-16,-1.000000e+00,Low,1.0,30.5,1645.0,1233617.0,10148.0,8572.0,4.0,1.100186


In [ ]:
df_merged.to_csv('master_merged_adjusted.csv', index=False)